# Results Summary - Table 3 and coverage curves

Aggregates the JSON files written by `run.py` into the per-configuration numbers of
**Table 3** (AUCC under Random / Frequency / DNF selection, final DNF coverage, and runtime),
and draws the coverage-vs-cost curves behind them.

Generate the inputs first:

```bash
bash scripts/run_table3.sh
```

For the whole table in one shot (no notebook needed) use `python scripts/make_table3.py`.

**Figure 3** (runtime scalability) comes from a separate benchmark:

```bash
bash scripts/gen_scalability_commands.sh
# execute the commands listed in all_commands_runtime.txt, then:
python scripts/plot_scalability.py
```

In [ ]:
# Bootstrap: run this notebook from either the repo root or notebooks/.
import os, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print('Working directory:', os.getcwd())

In [ ]:
import glob
import json
import os

import numpy as np

from src.utils import (
    calculate_auccc_from_results,
    plot_coverage_comparison,
    plot_violin_auccc,
    print_summary_table,
)

## 1. Discover the available result files

In [ ]:
FAMILY = 'neighbor_feature'   # or 'neighbor_only'

results_path = f'results/synthetic_{FAMILY}/'
results_files = sorted(glob.glob(os.path.join(results_path, '**', '*.json'), recursive=True))

if not results_files:
    raise FileNotFoundError(
        f'No results under {results_path}. Run `bash scripts/run_table3.sh` first.'
    )

for i, f in enumerate(results_files):
    print(f'[{i}] {os.path.basename(f)}')

## 2. Table 3 numbers, one configuration at a time

`print_summary_table` prints exactly the Table 3 columns for a single result file:
AUCC under Random / Frequency / DNF selection, the final DNF coverage, and the runtime,
each as mean +/- std over the `N` repetitions.

In [ ]:
for f in results_files:
    print_summary_table(f)
    print()

## 3. Coverage curves for one configuration

Pick an index from the listing in step 1.

In [ ]:
SELECTED = 0   # index into results_files

path = results_files[SELECTED]
print(path)

results = json.load(open(path))
summary = calculate_auccc_from_results(results)

plot_violin_auccc(summary)
plot_coverage_comparison(path)

## 4. Explanation time vs. graph size (from the trained-model runs)

This uses the `running_time` recorded by `run.py`, i.e. the same experiments as Table 3.
Figure 3 of the paper instead uses `run_test_runtime.py`, which skips training so the
measurement isolates explanation cost - see `scripts/plot_scalability.py`.

In [ ]:
import re
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

time_by_method_node = {}

for fpath in results_files:
    m = re.search(r'node(\d+)', os.path.basename(fpath))
    if not m:
        continue
    node_count = int(m.group(1))

    with open(fpath, 'r') as f:
        data = json.load(f)

    for run in data['runs']:
        for method, md in run['methods'].items():
            time_by_method_node.setdefault(method, {}).setdefault(node_count, []).append(
                md['running_time']
            )

fig, ax = plt.subplots(figsize=(10, 6))
colors = {'cf-greedy': '#2ca02c', 'cf': '#1f77b4', 'cff': '#ff7f0e'}
markers = {'cf-greedy': 's', 'cf': 'o', 'cff': '^'}
labels = {'cf-greedy': 'CF-Greedy (Ours)', 'cf': 'CF', 'cff': r'CF$^2$'}

for method in ['cf-greedy', 'cf', 'cff']:
    if method not in time_by_method_node:
        continue
    node_counts = sorted(time_by_method_node[method])
    means = [np.mean(time_by_method_node[method][n]) for n in node_counts]
    stds = [np.std(time_by_method_node[method][n]) for n in node_counts]
    ax.errorbar(node_counts, means, yerr=stds, label=labels[method],
                marker=markers[method], color=colors[method],
                capsize=4, linewidth=2, markersize=7)

ax.set_xscale('log', base=10)
ax.set_yscale('log')
ax.xaxis.set_major_formatter(ticker.ScalarFormatter())
ax.xaxis.set_minor_formatter(ticker.NullFormatter())
ax.set_xticks(sorted({n for m in time_by_method_node.values() for n in m}))
ax.get_xaxis().set_tick_params(which='minor', size=0)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda v, _: f'{v:g}'))
ax.set_xlabel('Number of Nodes (log scale)', fontsize=13)
ax.set_ylabel('Runtime (seconds, log scale)', fontsize=13)
ax.set_title(f'Runtime vs. Number of Nodes (synthetic {FAMILY})', fontsize=14)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()